# Appendix Tables

Generates per-model LaTeX tables for:
1. **Cumulative regret@10** — `high_scale` and `high_neg_scale`, one table per model per scale.
2. **Exploration count @3 and @10** — both scales, one table per model per scale.

In [1]:
import re
import os
import numpy as np
import pandas as pd

In [2]:
os.chdir("../..")

## Lookup Maps and Style Constants

In [3]:
rewards_per_scale = {
    'high_scale':     [75, 50, 25],
    'low_scale':      [0.75, 0.5, 0.25],
    'high_neg_scale': [-25, -50, -75],
    'low_neg_scale':  [-0.25, -0.5, -0.75],
}

nomenclature_map = {
    'alphanumeric':     'Alphanumeric',
    'ordinal_helpful':  'Ordinal - Helpful',
    'ordinal_mislead':  'Ordinal - Misleading',
    'world_helpful':    'World - Helpful',
    'world_mislead':    'World - Misleading',
    'sent_new_helpful': 'Sentiment - Helpful',
    'sent_new_mislead': 'Sentiment - Misleading',
}

## Load Data

In [4]:
results_df = pd.read_csv('experiments/merged_sem_var_results.csv')
print(f"Loaded {len(results_df)} rows")
results_df[['domain', 'model', 'history', 'variance']].value_counts().reset_index(name='count')

Loaded 1541 rows


,domain,model,history,variance,count
0,Farm,Llama3-8B,Summarized,Low,36
1,Farm,Qwen3-32B,Summarized,No,36
2,Farm,Qwen3-32B,Summarized,Low,36
3,Farm,Qwen3-32B,Summarized,High,36
4,Farm,Llama3-8B,Summarized,High,36
...,...,...,...,...,...
59,Classical,TS,Symbolic,No,2
60,Classical,UCB1,Symbolic,No,2
61,Classical,TS,Symbolic,Low,2
62,Classical,TS,Symbolic,High,2


## Configuration

In [5]:
SCALES = ['high_scale', 'high_neg_scale']
HIST   = 'Summarized'

MODELS_TO_INCLUDE = [
    ('Qwen3-32B',                    'Qwen3-32B'),
    ('Olmo-3.1-32B-Instruct',        'OLMo-3.1 32B'),
    ('gemini-3.1-flash-lite-preview', 'Gemini 3.1 Flash Lite'),
]

# Nomenclature rows: (display_label, helpful_key, mislead_key)
# None for mislead_key = no misleading counterpart
NOM_ROWS = [
    ('Ordinal',      'ordinal_helpful',  'ordinal_mislead'),
    ('World',        'world_helpful',    'world_mislead'),
    ('Sentiment',    'sent_new_helpful', 'sent_new_mislead'),
    ('Alphanumeric', 'alphanumeric',     None),
]

DOMAIN_ORDER   = ['Bandit', 'Farm', 'Clothing Recommendation']
DOMAIN_DISPLAY = {'Bandit': 'Bandit', 'Farm': 'Farm', 'Clothing Recommendation': 'Clothing'}
VARIANCE_ORDER = ['No', 'Low', 'High']

# Turn indices: index 2 = 3rd action (@3); index 9 = 10th/last action (@10)
EXPLOR_TURNS = [2, 9]

## Part 1: Cumulative Regret Tables

One table per model per scale. Rows = (variance, nomenclature). Columns = domain × {Helpful, Misleading}.

In [6]:
turn_numbers = sorted(
    int(col.split('_')[-2])
    for col in results_df.columns
    if col.startswith('cum_regret_') and col.endswith('_mean')
)
last_turn = max(turn_numbers)


def fmt(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return '--'
    return f'{val:.2f}'


def build_regret_table(df, present_domains, scale, model_label):
    """Build a per-model regret table (no Model column; Variance always shown)."""
    normalizing_constant = (
        max(rewards_per_scale[scale]) - min(rewards_per_scale[scale])
    ) * len(turn_numbers)

    # n_meta = 2: Variance, Nomenclature
    col_spec = 'lr' + ''.join(['|rr' for _ in present_domains])

    # Bold targets per (domain, variance): min H, max M
    llm_df = df[df['Nomenclature'] != '']
    bold_H, bold_M = {}, {}
    for domain in present_domains:
        dcol = DOMAIN_DISPLAY[domain]
        for variance in llm_df['Variance'].unique():
            mdf    = llm_df[llm_df['Variance'] == variance]
            h_vals = mdf[f'{dcol} H'].dropna()
            m_vals = mdf[f'{dcol} M'].dropna()
            if not h_vals.empty:
                bold_H[(dcol, variance)] = h_vals.min()
            if not m_vals.empty:
                bold_M[(dcol, variance)] = m_vals.max()

    def fmt_bold(val, target):
        s = fmt(val)
        if s == '--' or target is None:
            return s
        if abs(float(val) - target) < 1e-9:
            return r'\textbf{' + s + r'}'
        return s

    lines = []
    lines.append(r'\resizebox{\columnwidth}{!}{%')
    lines.append(r'\begin{tabular}{' + col_spec + r'}')
    lines.append(r'\toprule')

    # Header row 1: domain group spans
    header1 = ['', '']
    for domain in present_domains:
        header1.append(r'\multicolumn{2}{c}{' + DOMAIN_DISPLAY[domain] + r'}')
    lines.append(' & '.join(header1) + r' \\')

    # Cmidrules
    cmidrules = []
    for i in range(len(present_domains)):
        c1 = 3 + i * 2
        c2 = c1 + 1
        cmidrules.append(r'\cmidrule(lr){' + f'{c1}-{c2}' + r'}')
    lines.append(' '.join(cmidrules))

    # Header row 2
    header2 = ['Variance', 'Nomenclature']
    for _ in present_domains:
        header2 += ['Helpful', 'Misleading']
    lines.append(' & '.join(header2) + r' \\')
    lines.append(r'\midrule')

    prev_variance = None
    for _, row in df.iterrows():
        is_new_variance = (row['Variance'] != prev_variance)
        if prev_variance is not None and is_new_variance:
            lines.append(r'\midrule')

        variance_cell = row['Variance'] if is_new_variance else ''
        prev_variance = row['Variance']

        cells = [variance_cell, row['Nomenclature']]
        for domain in present_domains:
            dcol  = DOMAIN_DISPLAY[domain]
            h_val = row.get(f'{dcol} H')
            m_val = row.get(f'{dcol} M')
            key   = (dcol, row['Variance'])
            cells.append(fmt_bold(h_val, bold_H.get(key)))
            cells.append(fmt_bold(m_val, bold_M.get(key)))
        lines.append(' & '.join(cells) + r' \\')

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}%')
    lines.append(r'}')  # closes \resizebox
    return '\n'.join(lines)


def wrap_regret_table(tabular_str, model_label, scale):
    scale_note  = scale.replace('_', r'\_')
    model_safe  = model_label.replace(' ', '_').replace('.', '')
    return '\n'.join([
        r'\begin{table}[ht]',
        r'\centering',
        r'\small',
        tabular_str,
        r'\caption{Normalized cumulative regret at the final turn for ' + model_label
        + r', by variance condition and nomenclature type. '
        + r'H\,=\,helpful framing; M\,=\,misleading framing. Scale: ' + scale_note + r'.}',
        r'\label{tab:regret_' + model_safe + '_' + scale + r'}',
        r'\end{table}',
    ])


os.makedirs('figures/tables', exist_ok=True)

for model_id, model_label in MODELS_TO_INCLUDE:
    for scale in SCALES:
        normalizing_constant = (
            max(rewards_per_scale[scale]) - min(rewards_per_scale[scale])
        ) * len(turn_numbers)

        mask = (
            (results_df['history'] == HIST) &
            (results_df['scale']   == scale) &
            (results_df['model']   == model_id)
        )
        df = results_df[mask].copy()
        present_domains = [d for d in DOMAIN_ORDER if d in df['domain'].unique()]

        rows = []
        for variance in VARIANCE_ORDER:
            for nom_label, helpful_key, mislead_key in NOM_ROWS:
                row    = {'Variance': variance, 'Nomenclature': nom_label}
                subset = df[df['variance'] == variance]
                for domain in present_domains:
                    dcol       = DOMAIN_DISPLAY[domain]
                    dom_subset = subset[subset['domain'] == domain]
                    for col_suffix, key in [('H', helpful_key), ('M', mislead_key)]:
                        if key is None:
                            row[f'{dcol} {col_suffix}'] = float('nan')
                            continue
                        nom_df = dom_subset[dom_subset['nomenclature'].str.contains(key, na=False)]
                        if not nom_df.empty:
                            val = nom_df[f'cum_regret_{last_turn}_mean'].values[0]
                            row[f'{dcol} {col_suffix}'] = val / normalizing_constant
                        else:
                            row[f'{dcol} {col_suffix}'] = float('nan')
                rows.append(row)

        table_df = pd.DataFrame(rows)
        tabular  = build_regret_table(table_df, present_domains, scale, model_label)
        latex    = wrap_regret_table(tabular, model_label, scale)

        path = f'figures/tables/regret_table_{model_id}_{scale}.tex'
        with open(path, 'w') as f:
            f.write(latex)
        print(f'Saved: {path}')

print('\nDone.')

Saved: figures/tables/regret_table_Qwen3-32B_high_scale.tex
Saved: figures/tables/regret_table_Qwen3-32B_high_neg_scale.tex
Saved: figures/tables/regret_table_Olmo-3.1-32B-Instruct_high_scale.tex
Saved: figures/tables/regret_table_Olmo-3.1-32B-Instruct_high_neg_scale.tex
Saved: figures/tables/regret_table_gemini-3.1-flash-lite-preview_high_scale.tex
Saved: figures/tables/regret_table_gemini-3.1-flash-lite-preview_high_neg_scale.tex

Done.


## Part 2: Exploration Count Tables

One table per model per scale. Rows = (variance, nomenclature) + UCB1/TS baselines.  
Columns per domain: H@3 | H@10 | M@3 | M@10.

In [7]:
def fmt_count(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return '--'
    return f'{val:.1f}'


def build_exploration_table(df, present_domains, model_label, scale):
    """Build a per-model exploration-count table with turns @3 and @10 as columns.

    Column structure per domain: H@3 | H@10 | M@3 | M@10.
    """
    t3, t10 = EXPLOR_TURNS

    # n_meta = 2: Variance, Nomenclature
    col_spec = 'lr' + ''.join(['|rrrr' for _ in present_domains])

    # Bold targets per (domain, variance): max H@10, max M@10 (exclude classical baselines)
    llm_df = df[~df['Nomenclature'].isin(['UCB1', 'TS', ''])]
    bold_H10, bold_M10 = {}, {}
    for domain in present_domains:
        dcol = DOMAIN_DISPLAY[domain]
        for variance in llm_df['Variance'].unique():
            mdf     = llm_df[llm_df['Variance'] == variance]
            h10vals = mdf[f'{dcol} H@10'].dropna()
            m10vals = mdf[f'{dcol} M@10'].dropna()
            if not h10vals.empty:
                bold_H10[(dcol, variance)] = h10vals.max()
            if not m10vals.empty:
                bold_M10[(dcol, variance)] = m10vals.max()

    def fmt_bold_count(val, target):
        s = fmt_count(val)
        if s == '--' or target is None:
            return s
        if abs(float(val) - target) < 1e-9:
            return r'\textbf{' + s + r'}'
        return s

    lines = []
    lines.append(r'\resizebox{\columnwidth}{!}{%')
    lines.append(r'\begin{tabular}{' + col_spec + r'}')
    lines.append(r'\toprule')

    # Header row 1: domain group spans (4 cols each)
    header1 = ['', '']
    for domain in present_domains:
        header1.append(r'\multicolumn{4}{c}{' + DOMAIN_DISPLAY[domain] + r'}')
    lines.append(' & '.join(header1) + r' \\')

    # Cmidrules for domain groups
    dom_cmidrules = []
    for i in range(len(present_domains)):
        c1 = 3 + i * 4
        c2 = c1 + 3
        dom_cmidrules.append(r'\cmidrule(lr){' + f'{c1}-{c2}' + r'}')
    lines.append(' '.join(dom_cmidrules))

    # Header row 2: Helpful (span 2) | Misleading (span 2) per domain
    header2 = ['', '']
    hm_cmidrules = []
    for i, domain in enumerate(present_domains):
        c_h1 = 3 + i * 4
        c_h2 = c_h1 + 1
        c_m1 = c_h2 + 1
        c_m2 = c_m1 + 1
        header2.append(r'\multicolumn{2}{c}{Helpful}')
        header2.append(r'\multicolumn{2}{c}{Misleading}')
        hm_cmidrules.append(r'\cmidrule(lr){' + f'{c_h1}-{c_h2}' + r'}')
        hm_cmidrules.append(r'\cmidrule(lr){' + f'{c_m1}-{c_m2}' + r'}')
    lines.append(' & '.join(header2) + r' \\')
    lines.append(' '.join(hm_cmidrules))

    # Header row 3: @3 | @10 per framing per domain
    header3 = ['Variance', 'Nomenclature']
    for _ in present_domains:
        header3 += [r'@3', r'@10', r'@3', r'@10']
    lines.append(' & '.join(header3) + r' \\')
    lines.append(r'\midrule')

    prev_variance = None
    for _, row in df.iterrows():
        is_new_variance = (row['Variance'] != prev_variance)
        is_classical    = row['Nomenclature'] in ['UCB1', 'TS', '']

        if prev_variance is not None and is_new_variance:
            lines.append(r'\midrule')

        variance_cell = row['Variance'] if is_new_variance else ''
        prev_variance = row['Variance']

        cells = [variance_cell, row['Nomenclature']]
        for domain in present_domains:
            dcol = DOMAIN_DISPLAY[domain]
            if is_classical:
                cells.append(fmt_count(row.get(f'{dcol} H@3')))
                cells.append(fmt_count(row.get(f'{dcol} H@10')))
                cells.append(fmt_count(row.get(f'{dcol} M@3')))
                cells.append(fmt_count(row.get(f'{dcol} M@10')))
            else:
                key = (dcol, row['Variance'])
                cells.append(fmt_count(row.get(f'{dcol} H@3')))
                cells.append(fmt_bold_count(row.get(f'{dcol} H@10'), bold_H10.get(key)))
                cells.append(fmt_count(row.get(f'{dcol} M@3')))
                cells.append(fmt_bold_count(row.get(f'{dcol} M@10'), bold_M10.get(key)))
        lines.append(' & '.join(cells) + r' \\')

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}%')
    lines.append(r'}')  # closes \resizebox
    return '\n'.join(lines)


def wrap_exploration_table(tabular_str, model_label, scale):
    scale_note = scale.replace('_', r'\_')
    model_safe = model_label.replace(' ', '_').replace('.', '')
    return '\n'.join([
        r'\begin{table}[ht]',
        r'\centering',
        r'\small',
        tabular_str,
        r'\caption{Exploration count (unique arms tried) at turns 3 and 10 for ' + model_label
        + r', by variance condition and nomenclature type. '
        + r'H\,=\,helpful framing; M\,=\,misleading framing. Scale: ' + scale_note + r'.}',
        r'\label{tab:exploration_' + model_safe + '_' + scale + r'}',
        r'\end{table}',
    ])


baseline_df = pd.read_csv('experiments/merged_sem_var_results.csv')

for model_id, model_label in MODELS_TO_INCLUDE:
    for scale in SCALES:
        t3, t10 = EXPLOR_TURNS

        mask = (
            (results_df['history'] == HIST) &
            (results_df['scale']   == scale) &
            (results_df['model']   == model_id)
        )
        df = results_df[mask].copy()
        present_domains = [d for d in DOMAIN_ORDER if d in df['domain'].unique()]

        rows = []
        for variance in VARIANCE_ORDER:
            for nom_label, helpful_key, mislead_key in NOM_ROWS:
                row    = {'Variance': variance, 'Nomenclature': nom_label}
                subset = df[df['variance'] == variance]
                for domain in present_domains:
                    dcol       = DOMAIN_DISPLAY[domain]
                    dom_subset = subset[subset['domain'] == domain]
                    for col_suffix, turn_idx, key in [
                        ('H@3',  t3,  helpful_key),
                        ('H@10', t10, helpful_key),
                        ('M@3',  t3,  mislead_key),
                        ('M@10', t10, mislead_key),
                    ]:
                        if key is None:
                            row[f'{dcol} {col_suffix}'] = float('nan')
                            continue
                        nom_df = dom_subset[dom_subset['nomenclature'].str.contains(key, na=False)]
                        if not nom_df.empty:
                            row[f'{dcol} {col_suffix}'] = nom_df[f'exploration_count_{turn_idx}_mean'].values[0]
                        else:
                            row[f'{dcol} {col_suffix}'] = float('nan')
                rows.append(row)

        # Classical baselines (UCB1, TS) — domain='Classical', same value replicated across domains
        for bl_model, bl_label in [('UCB1', 'UCB1'), ('TS', 'TS')]:
            for variance in VARIANCE_ORDER:
                bl_row_df = baseline_df[
                    (baseline_df['model']        == bl_model) &
                    (baseline_df['variance']     == variance) &
                    (baseline_df['scale']        == scale) &
                    (baseline_df['nomenclature'] == 'baseline')
                ]
                row = {'Variance': variance, 'Nomenclature': bl_label}
                for domain in present_domains:
                    dcol = DOMAIN_DISPLAY[domain]
                    for col_suffix, turn_idx in [('H@3', t3), ('H@10', t10)]:
                        if not bl_row_df.empty:
                            row[f'{dcol} {col_suffix}'] = bl_row_df[f'exploration_count_{turn_idx}_mean'].values[0]
                        else:
                            row[f'{dcol} {col_suffix}'] = float('nan')
                    row[f'{dcol} M@3']  = float('nan')
                    row[f'{dcol} M@10'] = float('nan')
                rows.append(row)

        table_df = pd.DataFrame(rows)
        tabular  = build_exploration_table(table_df, present_domains, model_label, scale)
        latex    = wrap_exploration_table(tabular, model_label, scale)

        path = f'figures/tables/exploration_count_table_{model_id}_{scale}.tex'
        with open(path, 'w') as f:
            f.write(latex)
        print(f'Saved: {path}')

print('\nDone.')

Saved: figures/tables/exploration_count_table_Qwen3-32B_high_scale.tex
Saved: figures/tables/exploration_count_table_Qwen3-32B_high_neg_scale.tex
Saved: figures/tables/exploration_count_table_Olmo-3.1-32B-Instruct_high_scale.tex
Saved: figures/tables/exploration_count_table_Olmo-3.1-32B-Instruct_high_neg_scale.tex
Saved: figures/tables/exploration_count_table_gemini-3.1-flash-lite-preview_high_scale.tex
Saved: figures/tables/exploration_count_table_gemini-3.1-flash-lite-preview_high_neg_scale.tex

Done.
